In [1]:
CONFIG = dict(
    # ── Mode ─────────────────────────────────────────────────────────────────
    # 'train'   → train only
    # 'analyze' → load best_checkpoint.pth and run all analysis
    # 'both'    → train, then run analysis
    mode = 'both',
 
    # ── Paths ─────────────────────────────────────────────────────────────────
    data_path  = './data',
    output_dir = './dino_outputs',
 
    # ── ViT Architecture ──────────────────────────────────────────────────────
    # ViT-Tiny (fits in ~6 GB VRAM with batch_size=128)
    # patch_size=8 → 12×12=144 patches for 96×96 STL-10 images
    patch_size    = 8,
    embed_dim     = 192,   # ViT-Tiny=192, ViT-Small=384
    depth         = 12,    # number of transformer blocks
    num_heads     = 3,     # attention heads (embed_dim must be divisible)
 
    # ── Projection Head ───────────────────────────────────────────────────────
    # Paper uses out_dim=65536; we use 4096 to save memory
    out_dim        = 4096,
    hidden_dim     = 2048,
    bottleneck_dim = 256,
 
    # ── Multi-crop ────────────────────────────────────────────────────────────
    local_crops_number = 6,   # + 2 global = 8 crops total
 
    # ── Training ──────────────────────────────────────────────────────────────
    epochs      = 100,
    batch_size  = 128,    # reduce to 64 if OOM
    num_workers = 4,
 
    # ── Optimizer (AdamW) ─────────────────────────────────────────────────────
    lr               = 0.0005,   # scaled by batch_size/256 internally
    min_lr           = 1e-6,
    weight_decay     = 0.04,
    weight_decay_end = 0.4,
    warmup_epochs    = 10,
 
    # ── Teacher / EMA ─────────────────────────────────────────────────────────
    momentum_teacher = 0.996,   # cosine-scheduled up to 1.0
 
    # ── Temperatures ─────────────────────────────────────────────────────────
    # student_temp: fixed, higher → softer student distribution
    # teacher_temp: low → sharp targets (prevents uniform collapse)
    # warmup_teacher_temp: start temperature for teacher warmup
    # warmup_teacher_temp_epochs: epochs to ramp teacher temp
    student_temp                = 0.1,
    teacher_temp                = 0.04,
    warmup_teacher_temp         = 0.04,
    warmup_teacher_temp_epochs  = 30,
 
    # ── Evaluation ────────────────────────────────────────────────────────────
    eval_every = 10,    # run k-NN every N epochs
    knn_k      = 20,
    knn_temp   = 0.07,
)

In [2]:
import os
import sys
import copy
import math
import warnings
warnings.filterwarnings('ignore')
 
import numpy as np
from PIL import Image
from pathlib import Path
 
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torchvision.datasets import STL10
from torch.utils.data import DataLoader

In [3]:
class PatchEmbed(nn.Module):
    """
    Splits image into non-overlapping patches and linearly projects each.
    Implemented as a single Conv2d with kernel=stride=patch_size.
    """
    def __init__(self, img_size=96, patch_size=8, in_chans=3, embed_dim=192):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_chans, embed_dim,
                              kernel_size=patch_size, stride=patch_size)
 
    def forward(self, x):
        x = self.proj(x)          # (B, embed_dim, H/p, W/p)
        x = x.flatten(2)          # (B, embed_dim, num_patches)
        return x.transpose(1, 2)  # (B, num_patches, embed_dim)

In [4]:
class Attention(nn.Module):
    """
    Multi-head self-attention.
    Stores attention weights during forward for visualization.
    """
    def __init__(self, dim, num_heads=8, qkv_bias=True,
                 attn_drop=0., proj_drop=0.):
        super().__init__()
        self.num_heads = num_heads
        self.scale = (dim // num_heads) ** -0.5
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)
        self.attn_weights = None  # populated during forward, used for visualization
 
    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads)
        q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(0)
 
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        self.attn_weights = attn.detach()  # stored for visualization
 
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj_drop(self.proj(x))

In [5]:
class MLP(nn.Module):
    """Feed-forward block: Linear → GELU → Linear."""
    def __init__(self, in_features, hidden_features, out_features, drop=0.):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_features),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(hidden_features, out_features),
            nn.Dropout(drop),
        )
 
    def forward(self, x):
        return self.net(x)

In [6]:
class Block(nn.Module):
    """Standard ViT block: LN → Attn → residual → LN → MLP → residual."""
    def __init__(self, dim, num_heads, mlp_ratio=4., qkv_bias=True):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, eps=1e-6)
        self.attn  = Attention(dim, num_heads, qkv_bias)
        self.norm2 = nn.LayerNorm(dim, eps=1e-6)
        self.mlp   = MLP(dim, int(dim * mlp_ratio), dim)
 
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

In [7]:
class VisionTransformer(nn.Module):
    """
    ViT backbone. Returns [CLS] token from last block.
 
    get_last_selfattention(x) returns attention maps from the final block —
    shape (B, num_heads, N+1, N+1). The row at index 0 is CLS→all-patches,
    which is what the paper visualizes as "emergent segmentation".
    """
    def __init__(self, img_size=96, patch_size=8, in_chans=3,
                 embed_dim=192, depth=12, num_heads=3, mlp_ratio=4.):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size, patch_size, in_chans, embed_dim)
        num_patches = self.patch_embed.num_patches
 
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.pos_drop  = nn.Dropout(p=0.)
        self.blocks    = nn.ModuleList([
            Block(embed_dim, num_heads, mlp_ratio) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim, eps=1e-6)
 
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.apply(self._init_weights)
 
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.zeros_(m.bias); nn.init.ones_(m.weight)
 
    @property
    def embed_dim(self):
        return self.pos_embed.shape[-1]
 
    def _interpolate_pos_embed(self, x):
        """
        Interpolate positional embeddings to match the number of patches in x.
 
        Why this is needed:
          pos_embed is built for the 'native' image size (96×96 → 144 patches).
          Local crops are 48×48 → only 36 patches. The sequence lengths differ,
          so we can't add pos_embed directly.
 
        Fix (from the DINO / DeiT codebase):
          Separate the CLS token pos embed from patch pos embeds.
          Reshape patch embeds to a 2D grid, bilinearly interpolate to the
          new grid size, then flatten and re-attach the CLS pos embed.
          This is equivalent to what the paper does for arbitrary-resolution inputs.
        """
        N_native = self.pos_embed.shape[1] - 1          # e.g. 144 for 96×96
        N_actual = x.shape[1]                            # e.g. 36  for 48×48
        if N_native == N_actual:
            return self.pos_embed                        # no interpolation needed
 
        # Split CLS and patch positional embeddings
        pos_cls   = self.pos_embed[:, :1, :]             # (1, 1, d)
        pos_patch = self.pos_embed[:, 1:, :]             # (1, N_native, d)
 
        # Native grid size (assumed square)
        h0 = w0 = int(math.sqrt(N_native))              # e.g. 12
        # Target grid size (assumed square)
        h1 = w1 = int(math.sqrt(N_actual))              # e.g. 6
 
        # Reshape to spatial, interpolate, flatten back
        d = pos_patch.shape[-1]
        pos_patch = pos_patch.reshape(1, h0, w0, d).permute(0, 3, 1, 2)  # (1,d,h0,w0)
        pos_patch = F.interpolate(pos_patch, size=(h1, w1),
                                  mode='bicubic', align_corners=False)
        pos_patch = pos_patch.permute(0, 2, 3, 1).reshape(1, h1 * w1, d)  # (1,N_actual,d)
 
        return torch.cat([pos_cls, pos_patch], dim=1)   # (1, N_actual+1, d)
 
    def _prepare(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        cls = self.cls_token.expand(B, -1, -1)
        pos = self._interpolate_pos_embed(x)             # handles any input size
        x = torch.cat([cls, x], dim=1) + pos
        return self.pos_drop(x)
 
    def forward(self, x):
        x = self._prepare(x)
        for blk in self.blocks:
            x = blk(x)
        return self.norm(x)[:, 0]  # [CLS] token only
 
    def get_last_selfattention(self, x):
        """Forward pass storing attention only in last block."""
        x = self._prepare(x)
        for blk in self.blocks:
            x = blk(x)
        return self.blocks[-1].attn.attn_weights  # (B, heads, N+1, N+1)

In [8]:

class DINOHead(nn.Module):
    def __init__(self, in_dim, out_dim, hidden_dim=2048, bottleneck_dim=256,
                 nlayers=3, norm_last_layer=True):
        super().__init__()
        # Build MLP
        layers = [nn.Linear(in_dim, hidden_dim), nn.GELU()]
        for _ in range(nlayers - 2):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.GELU()]
        layers.append(nn.Linear(hidden_dim, bottleneck_dim))
        self.mlp = nn.Sequential(*layers)
        self.apply(self._init_weights)
 
        # Weight-normalized final linear (no bias)
        self.last_layer = nn.utils.weight_norm(
            nn.Linear(bottleneck_dim, out_dim, bias=False)
        )
        self.last_layer.weight_g.data.fill_(1)
        if norm_last_layer:
            self.last_layer.weight_g.requires_grad = False  # freeze magnitude
 
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
 
    def forward(self, x):
        x = self.mlp(x)
        x = F.normalize(x, dim=-1, p=2)  # project onto unit sphere
        return self.last_layer(x)

In [9]:
class DataAugmentationDINO:
    def __init__(self, global_size=96, local_size=48,
                 global_scale=(0.4, 1.0), local_scale=(0.05, 0.4),
                 local_crops_number=6):
        self.local_crops_number = local_crops_number
 
        flip_jitter = T.Compose([
            T.RandomHorizontalFlip(0.5),
            T.RandomApply([T.ColorJitter(0.4, 0.4, 0.2, 0.1)], p=0.8),
            T.RandomGrayscale(p=0.2),
        ])
        normalize = T.Compose([
            T.ToTensor(),
            T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
        ])
 
        # Global crop 1: always blur
        self.g1 = T.Compose([
            T.RandomResizedCrop(global_size, scale=global_scale,
                                interpolation=T.InterpolationMode.BICUBIC),
            flip_jitter,
            T.RandomApply([T.GaussianBlur(9, sigma=(0.1, 2.0))], p=1.0),
            normalize,
        ])
        # Global crop 2: solarize + rare blur
        self.g2 = T.Compose([
            T.RandomResizedCrop(global_size, scale=global_scale,
                                interpolation=T.InterpolationMode.BICUBIC),
            flip_jitter,
            T.RandomApply([T.GaussianBlur(9, sigma=(0.1, 2.0))], p=0.1),
            T.RandomSolarize(128, p=0.2),
            normalize,
        ])
        # Local crops: smaller, occasional blur, no solarize
        self.local = T.Compose([
            T.RandomResizedCrop(local_size, scale=local_scale,
                                interpolation=T.InterpolationMode.BICUBIC),
            flip_jitter,
            T.RandomApply([T.GaussianBlur(5, sigma=(0.1, 2.0))], p=0.5),
            normalize,
        ])
 
    def __call__(self, img):
        return (
            [self.g1(img), self.g2(img)] +
            [self.local(img) for _ in range(self.local_crops_number)]
        )

In [10]:
class DINOLoss(nn.Module):
    def __init__(self, out_dim, ncrops, warmup_teacher_temp, teacher_temp,
                 warmup_teacher_temp_epochs, nepochs,
                 student_temp=0.1, center_momentum=0.9):
        super().__init__()
        self.student_temp = student_temp
        self.center_momentum = center_momentum
        self.ncrops = ncrops
        self.register_buffer("center", torch.zeros(1, out_dim))
 
        # Teacher temperature schedule: warm → cold
        # Starting cold immediately can cause instability early in training
        self.teacher_temp_schedule = np.concatenate([
            np.linspace(warmup_teacher_temp, teacher_temp, warmup_teacher_temp_epochs),
            np.full(nepochs - warmup_teacher_temp_epochs, teacher_temp),
        ])
 
    def forward(self, student_output, teacher_output, epoch):
        # Student: divide by temperature, chunk into per-crop pieces
        student_chunks = (student_output / self.student_temp).chunk(self.ncrops)
 
        # Teacher: center, sharpen, softmax — no gradient flows here
        tau_t = self.teacher_temp_schedule[epoch]
        teacher_prob = F.softmax(
            (teacher_output - self.center) / tau_t, dim=-1
        ).detach().chunk(2)  # 2 global crops
 
        loss, n = 0.0, 0
        for iq, q in enumerate(teacher_prob):
            for v, s in enumerate(student_chunks):
                if v == iq:
                    continue  # skip same-view pair
                loss += torch.sum(-q * F.log_softmax(s, dim=-1), dim=-1).mean()
                n += 1
 
        self.update_center(teacher_output)
        return loss / n
 
    @torch.no_grad()
    def update_center(self, teacher_output):
        """EMA update of center vector from current batch."""
        batch_center = teacher_output.mean(dim=0, keepdim=True)
        self.center = self.center * self.center_momentum + \
                      batch_center * (1 - self.center_momentum)

In [11]:
def cosine_schedule(start, end, total):
    t = np.arange(total)
    return end - (end - start) * (np.cos(np.pi * t / total) + 1) / 2

In [12]:
@torch.no_grad()
def ema_update(student, teacher, m):
    """θ_t ← m·θ_t + (1−m)·θ_s"""
    for ps, pt in zip(student.parameters(), teacher.parameters()):
        pt.data.mul_(m).add_(ps.data, alpha=1.0 - m)

In [13]:
def lr_schedule(base_lr, min_lr, total, warmup):
    t = np.arange(total)
    warmup_vals = base_lr * t / max(warmup, 1)
    cosine_vals = min_lr + 0.5 * (base_lr - min_lr) * (
        1 + np.cos(np.pi * (t - warmup) / max(total - warmup, 1))
    )
    return np.where(t < warmup, warmup_vals, cosine_vals)

In [14]:
@torch.no_grad()
def knn_evaluate(backbone, train_loader, test_loader,
                 k=20, T=0.07, num_classes=10, device='cuda'):
    backbone.eval()
 
    def extract(loader):
        fs, ls = [], []
        for imgs, lbls in loader:
            f = F.normalize(backbone(imgs.to(device)), dim=-1)
            fs.append(f.cpu()); ls.append(lbls)
        return torch.cat(fs), torch.cat(ls)
 
    print("    Extracting train features...", end=' ', flush=True)
    tf, tl = extract(train_loader); print("done")
    print("    Extracting test  features...", end=' ', flush=True)
    qf, ql = extract(test_loader);  print("done")
 
    correct, total = 0, 0
    for s in range(0, len(qf), 256):
        q   = qf[s:s+256].to(device)
        lbl = ql[s:s+256].to(device)
        sims = q @ tf.t().to(device)
        topk_sims, topk_idx = sims.topk(k, dim=-1)
        topk_lbl = tl[topk_idx.cpu()].to(device)
        weights  = (topk_sims / T).exp()
        scores   = torch.zeros(len(q), num_classes, device=device)
        scores.scatter_add_(1, topk_lbl, weights)
        correct += (scores.argmax(1) == lbl).sum().item()
        total   += len(lbl)
 
    return 100.0 * correct / total

In [15]:
 
def train(cfg, device):
    os.makedirs(cfg['output_dir'], exist_ok=True)
 
    # ── Dataset ───────────────────────────────────────────────────────────────
    print("\n[1/5] Dataset (STL-10)...")
    aug = DataAugmentationDINO(
        local_crops_number=cfg['local_crops_number']
    )
    dataset = STL10(cfg['data_path'], split='unlabeled', download=True, transform=aug)
    loader  = DataLoader(dataset, batch_size=cfg['batch_size'], shuffle=True,
                         num_workers=cfg['num_workers'], pin_memory=True, drop_last=True)
 
    eval_tf  = T.Compose([T.ToTensor(),
                          T.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225))])
    knn_train = DataLoader(STL10(cfg['data_path'],'train', download=True,transform=eval_tf),
                           batch_size=256, num_workers=cfg['num_workers'])
    knn_test  = DataLoader(STL10(cfg['data_path'],'test',  download=True,transform=eval_tf),
                           batch_size=256, num_workers=cfg['num_workers'])
    print(f"  unlabeled={len(dataset)}  knn_train={len(knn_train.dataset)}  "
          f"knn_test={len(knn_test.dataset)}")
 
    # ── Models ────────────────────────────────────────────────────────────────
    print("\n[2/5] Building student / teacher...")
    def make_vit():
        return VisionTransformer(
            img_size=96, patch_size=cfg['patch_size'],
            embed_dim=cfg['embed_dim'], depth=cfg['depth'], num_heads=cfg['num_heads'],
        ).to(device)
 
    student_bb = make_vit()
    teacher_bb = copy.deepcopy(student_bb)
    student_hd = DINOHead(cfg['embed_dim'], cfg['out_dim'], cfg['hidden_dim'],
                          cfg['bottleneck_dim'], norm_last_layer=True).to(device)
    teacher_hd = DINOHead(cfg['embed_dim'], cfg['out_dim'], cfg['hidden_dim'],
                          cfg['bottleneck_dim'], norm_last_layer=False).to(device)
 
    for p in list(teacher_bb.parameters()) + list(teacher_hd.parameters()):
        p.requires_grad_(False)
 
    n = sum(p.numel() for p in student_bb.parameters() if p.requires_grad)
    print(f"  ViT-Tiny params: {n/1e6:.2f}M")
 
    # ── Loss ──────────────────────────────────────────────────────────────────
    ncrops = 2 + cfg['local_crops_number']
    loss_fn = DINOLoss(
        cfg['out_dim'], ncrops,
        cfg['warmup_teacher_temp'], cfg['teacher_temp'],
        cfg['warmup_teacher_temp_epochs'], cfg['epochs'],
        cfg['student_temp'],
    ).to(device)
 
    # ── Optimizer ─────────────────────────────────────────────────────────────
    print("\n[3/5] Optimizer + schedules...")
    # Separate WD=0 for biases and LayerNorm params
    reg, no_reg = [], []
    for m in [student_bb, student_hd]:
        for name, p in m.named_parameters():
            if not p.requires_grad: continue
            (no_reg if ('bias' in name or 'norm' in name) else reg).append(p)
    optimizer = torch.optim.AdamW(
        [{'params': reg}, {'params': no_reg, 'weight_decay': 0.0}],
        lr=cfg['lr'], weight_decay=cfg['weight_decay'],
    )
 
    total_iters  = cfg['epochs'] * len(loader)
    warmup_iters = cfg['warmup_epochs'] * len(loader)
    base_lr      = cfg['lr'] * cfg['batch_size'] / 256.0  # linear scaling rule
 
    lr_sched  = lr_schedule(base_lr, cfg['min_lr'], total_iters, warmup_iters)
    wd_sched  = cosine_schedule(cfg['weight_decay'], cfg['weight_decay_end'], total_iters)
    mom_sched = cosine_schedule(cfg['momentum_teacher'], 1.0, total_iters)
 
    print(f"  LR  : {base_lr:.2e} → {cfg['min_lr']:.2e}  (warmup {cfg['warmup_epochs']} ep)")
    print(f"  WD  : {cfg['weight_decay']} → {cfg['weight_decay_end']}")
    print(f"  EMA : {cfg['momentum_teacher']} → 1.0")
 
    # ── Loop ──────────────────────────────────────────────────────────────────
    print(f"\n[4/5] Training {cfg['epochs']} epochs  "
          f"(batch={cfg['batch_size']}, {ncrops} crops)...\n")
 
    log_path = os.path.join(cfg['output_dir'], 'dino_log.csv')
    with open(log_path, 'w') as f:
        f.write("epoch,loss,lr,wd,momentum,knn_acc\n")
 
    best_knn, global_it = 0.0, 0
 
    for epoch in range(cfg['epochs']):
        student_bb.train(); student_hd.train()
        epoch_loss = 0.0
 
        for bi, (crops, _) in enumerate(loader):
            # ── Update schedules ──────────────────────────────────────────────
            it = global_it
            for g in optimizer.param_groups:
                g['lr'] = lr_sched[it]
                if g['weight_decay'] > 0: g['weight_decay'] = wd_sched[it]
 
            imgs = [c.to(device, non_blocking=True) for c in crops]
 
            # ── Teacher forward (global crops only, no grad) ──────────────────
            with torch.no_grad():
                t_cls = torch.cat([teacher_bb(imgs[0]), teacher_bb(imgs[1])])
                t_out = teacher_hd(t_cls)
 
            # ── Student forward (separate passes for different crop sizes) ────
            # Global crops: 96×96, local crops: 48×48 → can't cat together
            s_global = student_bb(torch.cat(imgs[:2]))          # (2B, d)
            s_local  = student_bb(torch.cat(imgs[2:]))          # (6B, d)
            s_out    = student_hd(torch.cat([s_global, s_local]))
 
            # ── Loss + backward ───────────────────────────────────────────────
            loss = loss_fn(s_out, t_out, epoch)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                list(student_bb.parameters()) + list(student_hd.parameters()),
                max_norm=3.0,
            )
            # Freeze last-layer gradients in first epoch (paper trick for stability)
            if epoch == 0:
                for p in student_hd.last_layer.parameters():
                    p.grad = None
            optimizer.step()
 
            # ── EMA teacher update ────────────────────────────────────────────
            m = mom_sched[it]
            ema_update(student_bb, teacher_bb, m)
            ema_update(student_hd, teacher_hd, m)
 
            epoch_loss += loss.item()
            global_it  += 1
 
            if bi % 20 == 0:
                print(f"  Ep {epoch+1:3d}/{cfg['epochs']}  "
                      f"[{bi:4d}/{len(loader)}]  "
                      f"loss={loss.item():.4f}  "
                      f"lr={lr_sched[it]:.2e}  "
                      f"τ_t={loss_fn.teacher_temp_schedule[epoch]:.4f}  "
                      f"m={m:.4f}")
 
        avg_loss = epoch_loss / len(loader)
 
        # ── k-NN eval ─────────────────────────────────────────────────────────
        knn_acc = 0.0
        if (epoch + 1) % cfg['eval_every'] == 0 or epoch == cfg['epochs'] - 1:
            print(f"\n  → k-NN evaluation (epoch {epoch+1})...")
            knn_acc = knn_evaluate(teacher_bb, knn_train, knn_test,
                                   k=cfg['knn_k'], T=cfg['knn_temp'],
                                   device=device)
            print(f"  → k-NN (k={cfg['knn_k']}): {knn_acc:.2f}%\n")
 
            if knn_acc > best_knn:
                best_knn = knn_acc
                torch.save({
                    'epoch': epoch,
                    'student_backbone': student_bb.state_dict(),
                    'teacher_backbone': teacher_bb.state_dict(),
                    'student_head':     student_hd.state_dict(),
                    'optimizer':        optimizer.state_dict(),
                    'knn_acc':          knn_acc,
                    'cfg':              cfg,
                }, os.path.join(cfg['output_dir'], 'best_checkpoint.pth'))
 
        with open(log_path, 'a') as f:
            f.write(f"{epoch+1},{avg_loss:.6f},{lr_sched[global_it-1]:.8f},"
                    f"{wd_sched[global_it-1]:.6f},{mom_sched[global_it-1]:.6f},"
                    f"{knn_acc:.2f}\n")
 
        print(f"  ── Epoch {epoch+1:3d} | loss={avg_loss:.4f} | "
              f"knn={knn_acc:.2f}% | best={best_knn:.2f}% ──\n")
 
    print(f"\n[4/5] Done. Best k-NN: {best_knn:.2f}%")
    return teacher_bb, best_knn

In [16]:
def load_teacher(cfg, device):
    ckpt = torch.load(
        os.path.join(cfg['output_dir'], 'best_checkpoint.pth'),
        map_location=device, weights_only=False,
    )
    bb = VisionTransformer(
        img_size=96, patch_size=cfg['patch_size'],
        embed_dim=cfg['embed_dim'], depth=cfg['depth'], num_heads=cfg['num_heads'],
    ).to(device)
    bb.load_state_dict(ckpt['teacher_backbone'])
    bb.eval()
    print(f"  Loaded checkpoint (epoch {ckpt['epoch']+1}, "
          f"k-NN={ckpt['knn_acc']:.2f}%)")
    return bb

In [17]:
@torch.no_grad()
def extract_all(backbone, loader, device):
    fs, ls = [], []
    for imgs, lbls in loader:
        f = F.normalize(backbone(imgs.to(device)), dim=-1)
        fs.append(f.cpu()); ls.append(lbls)
    return torch.cat(fs), torch.cat(ls)

In [18]:
def run_knn_sweep(backbone, train_loader, test_loader, device):
    """Table 3 analog: sweep k to find optimal."""
    print("\n── k-NN sweep ──────────────────────────────────────────")
    print(f"  {'k':>5}  {'acc':>8}")
    results = {}
    for k in (5, 10, 20, 50, 100):
        acc = knn_evaluate(backbone, train_loader, test_loader,
                           k=k, T=0.07, device=device)
        results[k] = acc
        print(f"  {k:>5}  {acc:>7.2f}%")
    return results

In [19]:
def run_linear_probe(backbone, train_loader, test_loader, device, epochs=30):
    """Table 2 analog: linear probing on frozen features."""
    print("\n── Linear probe ─────────────────────────────────────────")
    print("  Extracting features...", end=' ', flush=True)
    tf, tl = extract_all(backbone, train_loader, device)
    qf, ql = extract_all(backbone, test_loader, device)
    tf, tl, qf, ql = tf.to(device), tl.to(device), qf.to(device), ql.to(device)
    print("done")
 
    clf = nn.Linear(tf.shape[1], 10).to(device)
    opt = torch.optim.SGD(clf.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best = 0.0
 
    for ep in range(epochs):
        clf.train()
        perm = torch.randperm(len(tf))
        for s in range(0, len(tf), 256):
            idx = perm[s:s+256]
            loss = F.cross_entropy(clf(tf[idx]), tl[idx])
            opt.zero_grad(); loss.backward(); opt.step()
        sch.step()
 
        if (ep + 1) % 5 == 0:
            clf.eval()
            with torch.no_grad():
                acc = (clf(qf).argmax(1) == ql).float().mean().item() * 100
            best = max(best, acc)
            print(f"  ep {ep+1:3d}/{epochs}  test_acc={acc:.2f}%")
 
    print(f"  Best linear probe accuracy: {best:.2f}%")
    return best

In [20]:
def visualize_attention(backbone, img_tensor, patch_size, output_path):
    """
    Visualize CLS→patch attention from last ViT block (Fig. 6 of paper).
    Each head attends to a different semantic aspect of the image.
    """
    try:
        import matplotlib.pyplot as plt
        from skimage.transform import resize as sk_resize
    except ImportError:
        print("  (skipping attention viz: matplotlib/skimage not installed)")
        return
 
    backbone.eval()
    device = next(backbone.parameters()).device
    img = img_tensor.unsqueeze(0).to(device)
    attn = backbone.get_last_selfattention(img)  # (1, heads, N+1, N+1)
 
    nh = attn.shape[1]
    # CLS token row, patch columns: shape (heads, num_patches)
    attn = attn[0, :, 0, 1:].cpu().numpy()
    h = w = img.shape[-1] // patch_size  # 96//8 = 12
    attn = attn.reshape(nh, h, w)
    attn_up = np.stack([
        sk_resize(attn[i], (img.shape[-2], img.shape[-1]),
                  anti_aliasing=True, order=3)
        for i in range(nh)
    ])
 
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img_np = img_tensor.permute(1,2,0).numpy() * std + mean
    img_np = img_np.clip(0, 1)
 
    fig, axes = plt.subplots(1, nh + 1, figsize=(3*(nh+1), 3))
    axes[0].imshow(img_np); axes[0].set_title("input", fontsize=9); axes[0].axis('off')
    for i in range(nh):
        axes[i+1].imshow(img_np)
        axes[i+1].imshow(attn_up[i], alpha=0.6, cmap='inferno')
        axes[i+1].set_title(f"head {i}", fontsize=9)
        axes[i+1].axis('off')
    plt.suptitle("DINO self-attention (CLS→patches, last block)", fontsize=10)
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  saved → {output_path}")
 
 
def run_tsne(backbone, test_loader, device, output_path):
    """Feature space t-SNE — should show 10 clean clusters with no labels used."""
    try:
        from sklearn.manifold import TSNE
        import matplotlib.pyplot as plt
    except ImportError:
        print("  (skipping t-SNE: sklearn/matplotlib not installed)")
        return
 
    print("\n── t-SNE visualization ──────────────────────────────────")
    print("  Extracting features...", end=' ', flush=True)
    feats, labels = extract_all(backbone, test_loader, device)
    print("done")
 
    print("  Running t-SNE (perplexity=30, 1000 iters)...", end=' ', flush=True)
    coords = TSNE(2, perplexity=30, n_iter=1000, random_state=42).fit_transform(feats.numpy())
    print("done")
 
    stl_classes = ['airplane','bird','car','cat','deer',
                   'dog','horse','monkey','ship','truck']
    colors = plt.cm.tab10(np.linspace(0, 1, 10))
    fig, ax = plt.subplots(figsize=(9, 7))
    for c in range(10):
        mask = labels.numpy() == c
        ax.scatter(coords[mask,0], coords[mask,1], c=[colors[c]],
                   label=stl_classes[c], alpha=0.6, s=6, rasterized=True)
    ax.legend(fontsize=8, markerscale=2, loc='upper right')
    ax.set_title("t-SNE of DINO features (teacher, STL-10 test set)", fontsize=11)
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  saved → {output_path}")

In [21]:
def plot_curves(log_path, output_path):
    """Training loss + k-NN accuracy curves."""
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        return
    if not os.path.exists(log_path):
        return
 
    data = np.genfromtxt(log_path, delimiter=',', skip_header=1)
    if data.ndim == 1: data = data[None]
    epochs, loss, knn = data[:,0], data[:,1], data[:,5]
    mask = knn > 0
 
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
    a1.plot(epochs, loss, 'steelblue', lw=1.5)
    a1.set(xlabel='epoch', ylabel='loss', title='DINO training loss'); a1.grid(alpha=0.3)
    a2.plot(epochs[mask], knn[mask], 'seagreen', lw=1.5, marker='o', ms=5)
    a2.set(xlabel='epoch', ylabel='k-NN acc (%)',
           title=f'k-NN accuracy (k={CONFIG["knn_k"]}, STL-10)')
    a2.grid(alpha=0.3)
    plt.suptitle("DINO on STL-10", fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  saved → {output_path}")


In [22]:
def analyze(cfg, device):
    print("\n" + "="*60)
    print("[5/5] Post-training analysis")
    print("="*60)
 
    backbone = load_teacher(cfg, device)
 
    eval_tf  = T.Compose([T.ToTensor(),
                          T.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225))])
    knn_train = DataLoader(STL10(cfg['data_path'],'train',download=True,transform=eval_tf),
                           batch_size=256, num_workers=cfg['num_workers'])
    knn_test  = DataLoader(STL10(cfg['data_path'],'test', download=True,transform=eval_tf),
                           batch_size=256, num_workers=cfg['num_workers'])
 
    out = cfg['output_dir']
 
    # 1. k-NN sweep
    knn_results = run_knn_sweep(backbone, knn_train, knn_test, device)
 
    # 2. Linear probe
    lin_acc = run_linear_probe(backbone, knn_train, knn_test, device)
 
    # 3. Attention maps (8 test images)
    print("\n── Attention maps ───────────────────────────────────────")
    vis_tf = T.Compose([T.Resize(96), T.CenterCrop(96),
                        T.ToTensor(), T.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225))])
    vis_ds = STL10(cfg['data_path'], 'test', download=True, transform=vis_tf)
    for i in range(8):
        img, _ = vis_ds[i * 100]
        visualize_attention(backbone, img, cfg['patch_size'],
                            os.path.join(out, f'attn_{i}.png'))
 
    # 4. t-SNE
    run_tsne(backbone, knn_test, device, os.path.join(out, 'tsne.png'))
 
    # 5. Training curves
    plot_curves(os.path.join(out, 'dino_log.csv'), os.path.join(out, 'curves.png'))
 
    # Summary
    best_k   = max(knn_results, key=knn_results.get)
    best_knn = knn_results[best_k]
    print(f"""
╔══════════════════════════════════════╗
║  DINO Analysis Summary               ║
╠══════════════════════════════════════╣
║  Best k-NN : {best_knn:.2f}% (k={best_k:<3d})          ║
║  Linear    : {lin_acc:.2f}%                  ║
║  Baseline  : 10.0% (random)          ║
║  Supervised ViT-Tiny upper bound: ~80%║
╚══════════════════════════════════════╝""")

In [23]:
def main():
    cfg    = CONFIG
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
 
    print("╔══════════════════════════════════════════════════════════╗")
    print("║  DINO on STL-10                                          ║")
    print("╚══════════════════════════════════════════════════════════╝")
    print(f"  device   : {device}" + (f" ({torch.cuda.get_device_name(0)})" if device.type=='cuda' else ''))
    print(f"  mode     : {cfg['mode']}")
    print(f"  arch     : ViT embed={cfg['embed_dim']} depth={cfg['depth']} "
          f"heads={cfg['num_heads']} patch={cfg['patch_size']}")
    print(f"  training : {cfg['epochs']} epochs, batch={cfg['batch_size']}, "
          f"crops=2+{cfg['local_crops_number']}")
    print(f"  τ_s={cfg['student_temp']}  τ_t={cfg['teacher_temp']}  "
          f"out_dim={cfg['out_dim']}")
 
    if cfg['mode'] in ('train', 'both'):
        train(cfg, device)
 
    if cfg['mode'] in ('analyze', 'both'):
        analyze(cfg, device)


In [ ]:
main()

╔══════════════════════════════════════════════════════════╗
║  DINO on STL-10                                          ║
╚══════════════════════════════════════════════════════════╝
  device   : cuda (Tesla T4)
  mode     : both
  arch     : ViT embed=192 depth=12 heads=3 patch=8
  training : 100 epochs, batch=128, crops=2+6
  τ_s=0.1  τ_t=0.04  out_dim=4096

[1/5] Dataset (STL-10)...


100%|██████████| 2.64G/2.64G [02:13<00:00, 19.8MB/s] 


  unlabeled=100000  knn_train=5000  knn_test=8000

[2/5] Building student / teacher...
  ViT-Tiny params: 5.40M

[3/5] Optimizer + schedules...
  LR  : 2.50e-04 → 1.00e-06  (warmup 10 ep)
  WD  : 0.04 → 0.4
  EMA : 0.996 → 1.0

[4/5] Training 100 epochs  (batch=128, 8 crops)...

  Ep   1/100  [   0/781]  loss=8.5105  lr=0.00e+00  τ_t=0.0400  m=0.9960
  Ep   1/100  [  20/781]  loss=8.5025  lr=6.40e-07  τ_t=0.0400  m=0.9960
  Ep   1/100  [  40/781]  loss=8.4760  lr=1.28e-06  τ_t=0.0400  m=0.9960
  Ep   1/100  [  60/781]  loss=8.4224  lr=1.92e-06  τ_t=0.0400  m=0.9960
  Ep   1/100  [  80/781]  loss=8.3877  lr=2.56e-06  τ_t=0.0400  m=0.9960
  Ep   1/100  [ 100/781]  loss=8.3475  lr=3.20e-06  τ_t=0.0400  m=0.9960
  Ep   1/100  [ 120/781]  loss=8.3067  lr=3.84e-06  τ_t=0.0400  m=0.9960
  Ep   1/100  [ 140/781]  loss=8.2948  lr=4.48e-06  τ_t=0.0400  m=0.9960
  Ep   1/100  [ 160/781]  loss=8.2825  lr=5.12e-06  τ_t=0.0400  m=0.9960
  Ep   1/100  [ 180/781]  loss=8.2478  lr=5.76e-06  τ_t=0.0400 